# Crack Dataset — Exploratory Data Analysis (EDA)

**Why this notebook exists.** Training silently used OmniCrack30k masks at the wrong polarity
(black crack on white bg) while assuming CrackSeg9k's polarity (white crack on black). ~77% of
labels became full-frame 'crack' polygons. The model metric (mask mAP 0.825) looked great because
it was scored against the *same* broken labels — metrics validate model-vs-label, never
label-vs-reality.

**The lesson:** verify data assumptions on real samples *before* building on them. This notebook is
the gate that would have caught it in 5 minutes — and a reusable template for any mask dataset.

Run top-to-bottom. Each section ends with a takeaway. The final cell is an automated red-flag checklist.


## 0. Setup


In [2]:
import sys, random
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Find repo root (the dir whose src/ holds config.py) and put src/ on path
ROOT = Path.cwd()
while not (ROOT / 'src' / 'config.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'config.py').exists(), 'Run this from inside the repo (src/config.py not found)'
sys.path.insert(0, str(ROOT / 'src'))

from config import CRACKSEG9K, OMNICRACK30K, MIN_CRACK_PIXELS
from convert_annotations import mask_to_yolo_polygons, _contour_pixel_count
from eval_mask_conversion import polygons_to_mask

# config paths are relative to the repo root; the notebook's cwd is notebooks/,
# so anchor them to ROOT (ROOT / abs_path stays absolute, so this is always safe).
CRACKSEG9K = ROOT / CRACKSEG9K
OMNICRACK30K = ROOT / OMNICRACK30K

random.seed(0); np.random.seed(0)
print('repo root :', ROOT)
print('MIN_CRACK_PIXELS =', MIN_CRACK_PIXELS)

repo root : /Users/johnnytan5/Documents/FYP/Model Training
MIN_CRACK_PIXELS = 15


In [3]:
# Dataset registry: (name, masks_dir, images_dir, invert)
# invert=True means crack is the DARK class and must be flipped to foreground.
DATASETS = [
    ('CrackSeg9k', CRACKSEG9K / 'masks', CRACKSEG9K / 'images', False),
    ('OmniCrack30k', OMNICRACK30K / 'annotations' / 'training',
                     OMNICRACK30K / 'images' / 'training', True),
]

def list_masks(mask_dir, n=None):
    fs = sorted(Path(mask_dir).glob('*.png'))
    if n and len(fs) > n:
        fs = random.sample(fs, n)
    return fs

def load_mask(path, invert):
    m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if m is None:
        return None
    return (255 - m) if invert else m  # crack = white after this

def find_image(images_dir, stem):
    for ext in ('.jpg', '.png', '.jpeg'):
        p = Path(images_dir) / f'{stem}{ext}'
        if p.exists():
            return p
    return None

for name, mdir, idir, inv in DATASETS:
    print(f'{name:13s} masks_dir exists={Path(mdir).exists()}  images_dir exists={Path(idir).exists()}  invert={inv}')

CrackSeg9k    masks_dir exists=True  images_dir exists=True  invert=False
OmniCrack30k  masks_dir exists=True  images_dir exists=True  invert=True


## 1. Inventory & image/mask parity

First sanity: do image and mask counts match per dataset/split? A mismatch = orphans = silent
drops downstream.


In [4]:
rows = []
# CrackSeg9k (flat)
rows.append(('CrackSeg9k', 'all',
             len(list((CRACKSEG9K/'images').glob('*'))),
             len(list((CRACKSEG9K/'masks').glob('*.png')))))
# OmniCrack30k (pre-split)
for sp in ('training', 'validation', 'test'):
    rows.append(('OmniCrack30k', sp,
                 len(list((OMNICRACK30K/'images'/sp).glob('*'))),
                 len(list((OMNICRACK30K/'annotations'/sp).glob('*.png')))))

print(f"{'dataset':14s}{'split':12s}{'images':>9s}{'masks':>9s}  parity")
for d, sp, ni, nm in rows:
    print(f'{d:14s}{sp:12s}{ni:9d}{nm:9d}  {"OK" if ni==nm else "MISMATCH!"}')

dataset       split          images    masks  parity
CrackSeg9k    all               250      250  OK
OmniCrack30k  training        22158    22158  OK
OmniCrack30k  validation       3277     3277  OK
OmniCrack30k  test             4582     4582  OK


## 1.5 Sample raw images — one per dataset

Before inspecting masks, look at one raw photo from each source. CrackSeg9k is close-up crack
texture; OmniCrack30k spans varied surfaces and viewpoints. This frames *what* the model sees.

In [5]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6*len(DATASETS), 5))
axes = np.atleast_1d(axes)
for c, (name, mdir, idir, inv) in enumerate(DATASETS):
    f = list_masks(mdir, 1)[0]                      # one random sample (seeded above)
    img_p = find_image(idir, f.stem)
    img = (cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
           if img_p is not None else cv2.imread(str(f)))
    axes[c].imshow(img)
    axes[c].set_title(f'{name}\n{f.name}  ({img.shape[1]}x{img.shape[0]})', fontsize=10)
    axes[c].axis('off')
plt.suptitle('Sample raw image — one per dataset'); plt.tight_layout(); plt.show()

<Figure size 1200x500 with 2 Axes>

## 2. Mask polarity & pixel-value distribution  — *the check we missed*

For a crack dataset, the crack is always a **small minority** of pixels. So if we threshold the
raw mask and the bright class covers the *majority*, the mask is **inverted** (crack is the dark
class). This one plot would have screamed the bug.


In [18]:
fig, axes = plt.subplots(len(DATASETS), 2, figsize=(12, 4*len(DATASETS)))
axes = np.atleast_2d(axes)
for r, (name, mdir, idir, inv) in enumerate(DATASETS):
    fs = list_masks(mdir, 300)
    raw_frac, vals = [], []
    for f in fs:
        m = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if m is None: continue
        raw_frac.append((m > 127).mean())      # RAW (no invert) — reveals polarity
        vals.append(m.ravel()[::97])           # subsample pixel values
    if not vals:
        print(f'{name}: no masks loaded from {mdir} — skipping (check path / data present)')
        axes[r,0].set_visible(False); axes[r,1].set_visible(False); continue
    raw_frac = np.array(raw_frac)
    vals = np.concatenate(vals)
    axes[r,0].hist(raw_frac, bins=40, color='steelblue')
    axes[r,0].set_title(f'{name}: RAW bright-pixel fraction per mask')
    axes[r,0].set_xlabel('fraction of pixels > 127'); axes[r,0].axvline(0.5, color='r', ls='--')
    axes[r,1].hist(vals, bins=50, color='slategray')
    axes[r,1].set_title(f'{name}: pixel value distribution')
    axes[r,1].set_xlabel('gray value 0..255')
    median = np.median(raw_frac)
    flag = 'INVERTED (crack=dark)' if median > 0.5 else 'normal (crack=bright)'
    print(f'{name:13s} median bright-frac = {median:.3f}  ->  {flag}   (declared invert={inv})')
plt.tight_layout(); plt.show()

CrackSeg9k    median bright-frac = 0.016  ->  normal (crack=bright)   (declared invert=False)
OmniCrack30k  median bright-frac = 0.992  ->  INVERTED (crack=dark)   (declared invert=True)


<Figure size 1200x800 with 4 Axes>

**Takeaway.** If `median bright-frac > 0.5`, the bright class is the *background* → crack is dark →
must invert. CrackSeg9k sits ~0.03 (crack is the bright minority, correct). OmniCrack30k sits ~0.97
(background is bright → inverted). The `declared invert` must match: CrackSeg9k=False, Omni=True.


## 3. Raw mask gallery — eyeball both datasets side by side

Numbers lie less when you also *look*. Raw masks, no correction: the polarity flip is obvious.


In [19]:
N = 4
fig, axes = plt.subplots(len(DATASETS), N, figsize=(3*N, 3*len(DATASETS)))
axes = np.atleast_2d(axes)
for r, (name, mdir, idir, inv) in enumerate(DATASETS):
    for c, f in enumerate(list_masks(mdir, N)):
        m = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        axes[r,c].imshow(m, cmap='gray', vmin=0, vmax=255)
        axes[r,c].set_title(f'{name}\n{f.name}', fontsize=8)
        axes[r,c].axis('off')
plt.suptitle('RAW masks (no polarity correction)'); plt.tight_layout(); plt.show()

<Figure size 1200x600 with 8 Axes>

## 4. Polarity-corrected mask + image overlay

Apply the per-dataset `invert`, then overlay the corrected crack (red) on the original photo.
The red should sit *on* the visible crack — confirms both polarity AND image↔mask alignment.


In [20]:
N = 3
fig, axes = plt.subplots(len(DATASETS), N, figsize=(4*N, 4*len(DATASETS)))
axes = np.atleast_2d(axes)
for r, (name, mdir, idir, inv) in enumerate(DATASETS):
    for c, f in enumerate(list_masks(mdir, N)):
        img_p = find_image(idir, f.stem)
        m = load_mask(f, inv)            # corrected: crack=white
        _, mb = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
        if img_p is not None:
            img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
            if img.shape[:2] != mb.shape[:2]:
                mb = cv2.resize(mb, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
            ov = img.copy(); ov[mb > 0] = [255, 0, 0]
            disp = cv2.addWeighted(img, 0.6, ov, 0.4, 0)
        else:
            disp = cv2.cvtColor(mb, cv2.COLOR_GRAY2RGB)
        axes[r,c].imshow(disp)
        axes[r,c].set_title(f'{name}: {f.name}', fontsize=8); axes[r,c].axis('off')
plt.suptitle('Corrected crack (red) overlaid on image'); plt.tight_layout(); plt.show()

<Figure size 1200x800 with 6 Axes>

## 5. Mask → polygon conversion: 3-panel comparison

The actual training labels are YOLO polygons derived from the mask. Show
`original | GT mask (corrected) | polygon overlay` so a crack present in the GT panel with no
polygon beside it = a dropped label.


In [21]:
def three_panel(img_p, mask_p, invert):
    img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB) if img_p else None
    m = load_mask(mask_p, invert)
    _, mb = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
    h, w = mb.shape
    if img is None:
        img = cv2.cvtColor(mb, cv2.COLOR_GRAY2RGB)
    if img.shape[:2] != (h, w):
        img = cv2.resize(img, (w, h))
    polys = mask_to_yolo_polygons(mask_p, invert=invert)
    ov = img.copy()
    for ln in polys:
        p = ln.split(); pts = (np.array(p[1:], float).reshape(-1,2) * [w,h]).astype(np.int32)
        cv2.polylines(ov, [pts], True, (0,255,0), 2); cv2.fillPoly(ov, [pts], (0,200,0))
    ov = cv2.addWeighted(img, 0.6, ov, 0.4, 0)
    return img, mb, ov, len(polys)

for name, mdir, idir, inv in DATASETS:
    fig, axes = plt.subplots(3, 3, figsize=(11, 11))
    for r, f in enumerate(list_masks(mdir, 3)):
        img, mb, ov, npoly = three_panel(find_image(idir, f.stem), f, inv)
        axes[r,0].imshow(img); axes[r,0].set_title('original', fontsize=9)
        axes[r,1].imshow(mb, cmap='gray'); axes[r,1].set_title('GT mask (crack=white)', fontsize=9)
        axes[r,2].imshow(ov); axes[r,2].set_title(f'polygons n={npoly}', fontsize=9)
        for a in axes[r]: a.axis('off')
    plt.suptitle(name); plt.tight_layout(); plt.show()

<Figure size 1100x1100 with 9 Axes>

<Figure size 1100x1100 with 9 Axes>

## 6. Miss-rate & contour-size distribution

How often does a crack-bearing mask produce an **empty** label (silent drop)? And how does the
`MIN_CRACK_PIXELS` threshold trade misses against tiny specks? Filtering on contour **pixel count**
(not enclosed area) keeps 1px hairlines at true width.


In [22]:
for name, mdir, idir, inv in DATASETS:
    fs = list_masks(mdir, 400)
    sizes, miss, has = [], 0, 0
    for f in fs:
        m = load_mask(f, inv)
        if (m > 127).sum() == 0:
            continue  # background image, not a miss
        has += 1
        _, b = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
        cnts, _ = cv2.findContours(b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        px = [_contour_pixel_count(c) for c in cnts]
        sizes += px
        if not any(p >= MIN_CRACK_PIXELS for p in px):
            miss += 1
    sizes = np.array(sizes)
    print(f'{name}: has_crack={has}  miss_rate={miss}/{has}={100*miss/max(has,1):.2f}% at MIN_CRACK_PIXELS={MIN_CRACK_PIXELS}')
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].hist(sizes[sizes < 300], bins=60, color='teal')
    ax[0].axvline(MIN_CRACK_PIXELS, color='r', ls='--', label=f'cutoff={MIN_CRACK_PIXELS}')
    ax[0].set_title(f'{name}: contour pixel-count (<300)'); ax[0].set_xlabel('pixels'); ax[0].legend()
    thr = [5, 10, 15, 20, 30, 50]
    mr = []
    for t in thr:
        mm = hh = 0
        for f in fs:
            m = load_mask(f, inv)
            if (m > 127).sum() == 0: continue
            hh += 1
            _, b = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
            cnts, _ = cv2.findContours(b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if not any(_contour_pixel_count(c) >= t for c in cnts): mm += 1
        mr.append(100*mm/max(hh,1))
    ax[1].plot(thr, mr, 'o-'); ax[1].set_title(f'{name}: miss-rate vs threshold')
    ax[1].set_xlabel('MIN_CRACK_PIXELS'); ax[1].set_ylabel('miss-rate %')
    plt.tight_layout(); plt.show()

CrackSeg9k: has_crack=250  miss_rate=0/250=0.00% at MIN_CRACK_PIXELS=15


<Figure size 1200x350 with 2 Axes>

OmniCrack30k: has_crack=341  miss_rate=7/341=2.05% at MIN_CRACK_PIXELS=15


<Figure size 1200x350 with 2 Axes>

## 7. Crack geometry — how thin are these cracks?

Distribution of crack-pixel fraction, contours per image, and a crude width estimate
(`width ~ 2*area / perimeter`). Confirms why mAP is the wrong metric: cracks are 1–few px wide,
so a 1px offset destroys instance IoU.


In [23]:
for name, mdir, idir, inv in DATASETS:
    fracs, ncnts, widths = [], [], []
    for f in list_masks(mdir, 400):
        m = load_mask(f, inv)
        _, b = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
        fr = (b > 0).mean()
        if fr == 0: continue
        fracs.append(fr)
        cnts, _ = cv2.findContours(b, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = [c for c in cnts if _contour_pixel_count(c) >= MIN_CRACK_PIXELS]
        ncnts.append(len(cnts))
        for c in cnts:
            per = cv2.arcLength(c, True)
            area = _contour_pixel_count(c)
            if per > 0: widths.append(2*area/per)  # ~ width of a thin ribbon
    if not fracs:
        print(f'{name}: no crack masks loaded — skipping geometry'); continue
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
    ax[0].hist(fracs, bins=40, color='darkorange'); ax[0].set_title(f'{name}: crack pixel fraction')
    ax[1].hist(ncnts, bins=range(0, 15), color='seagreen'); ax[1].set_title('cracks per image')
    ax[2].hist(np.clip(widths, 0, 20), bins=40, color='indianred'); ax[2].set_title('est. crack width (px)')
    plt.tight_layout(); plt.show()
    mw = np.median(widths) if widths else float('nan')
    mc = int(np.median(ncnts)) if ncnts else 0
    print(f'{name}: median width ~ {mw:.1f}px  | median cracks/img = {mc}')

<Figure size 1500x320 with 3 Axes>

CrackSeg9k: median width ~ 3.0px  | median cracks/img = 2


<Figure size 1500x320 with 3 Axes>

OmniCrack30k: median width ~ 3.0px  | median cracks/img = 1


## 8. Image resolution & aspect ratio per dataset

Native resolution caps how much thin-crack detail exists. Training imgsz upscales these; you can't
recover detail beyond the source.


In [24]:
for name, mdir, idir, inv in DATASETS:
    hs, ws = [], []
    for f in list_masks(mdir, 300):
        ip = find_image(idir, f.stem)
        im = cv2.imread(str(ip)) if ip else cv2.imread(str(f))
        if im is None: continue
        h, w = im.shape[:2]; hs.append(h); ws.append(w)
    hs, ws = np.array(hs), np.array(ws)
    print(f'{name}: H {hs.min()}-{hs.max()} (med {int(np.median(hs))})  '
          f'W {ws.min()}-{ws.max()} (med {int(np.median(ws))})  n={len(hs)}')

CrackSeg9k: H 400-400 (med 400)  W 400-400 (med 400)  n=250
OmniCrack30k: H 105-1440 (med 256)  W 224-2560 (med 256)  n=300


## 9. Red-flag checklist (automated)

Run this as a **pre-training gate**. Any FAIL = stop and inspect before converting/training.


In [25]:
def check(name, ok, detail=''):
    print(f'[{"PASS" if ok else "FAIL"}] {name}' + (f'  -> {detail}' if detail else ''))
    return ok

all_ok = True
for name, mdir, idir, inv in DATASETS:
    fs = list_masks(mdir, 200)
    bright = np.median([(cv2.imread(str(f),0) > 127).mean() for f in fs])
    declared_inverted = inv
    detected_inverted = bright > 0.5
    all_ok &= check(f'{name}: polarity declaration matches data',
                    declared_inverted == detected_inverted,
                    f'declared invert={inv}, bright-frac={bright:.2f} (>0.5 means dark crack)')
    # crack is minority after correction
    corr = np.median([( load_mask(f, inv) > 127).mean() for f in fs])
    all_ok &= check(f'{name}: crack is minority class after correction', corr < 0.3,
                    f'corrected crack fraction={corr:.3f}')
    # parity
    ni = len(list(Path(idir).glob('*'))); nm = len(list(Path(mdir).glob('*.png')))
    all_ok &= check(f'{name}: image/mask count parity', ni == nm, f'{ni} imgs vs {nm} masks')
    # miss-rate
    miss=has=0
    for f in fs:
        m = load_mask(f, inv)
        if (m>127).sum()==0: continue
        has+=1
        if len(mask_to_yolo_polygons(f, invert=inv))==0: miss+=1
    mr = 100*miss/max(has,1)
    all_ok &= check(f'{name}: conversion miss-rate < 5%', mr < 5, f'{mr:.1f}%')

print()
print('ALL CHECKS PASSED — safe to convert/train.' if all_ok else 'SOME CHECKS FAILED — inspect before training.')

[PASS] CrackSeg9k: polarity declaration matches data  -> declared invert=False, bright-frac=0.02 (>0.5 means dark crack)
[PASS] CrackSeg9k: crack is minority class after correction  -> corrected crack fraction=0.016
[PASS] CrackSeg9k: image/mask count parity  -> 250 imgs vs 250 masks
[PASS] CrackSeg9k: conversion miss-rate < 5%  -> 0.0%
[PASS] OmniCrack30k: polarity declaration matches data  -> declared invert=True, bright-frac=0.99 (>0.5 means dark crack)
[PASS] OmniCrack30k: crack is minority class after correction  -> corrected crack fraction=0.011
[PASS] OmniCrack30k: image/mask count parity  -> 22158 imgs vs 22158 masks
[PASS] OmniCrack30k: conversion miss-rate < 5%  -> 2.4%

ALL CHECKS PASSED — safe to convert/train.
